In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
from functions import get_user_number_from_config, data_from_data_sink

In [ ]:
user = get_user_number_from_config()
print(user)

In [ ]:
query = f"""
SELECT 
    steps_avg,
    sleep_hours_avg,
    sleep_hours_std,
    sleep_hours_goal,
    sleep_wakeups_avg,
    tot_wellness_score
FROM dim_health
WHERE user_number = {user}
Order by entity_id DESC
LIMIT 1;
"""
df_health = data_from_data_sink(query)
print(df_health)

In [ ]:
query = f"""
SELECT 
    at_id
FROM dim_health
WHERE user_number = {user}
ORDER BY entity_id DESC
LIMIT 1;
"""
at_id= data_from_data_sink(query)
print(at_id)
at_id = int(at_id["at_id"][0])
print(at_id)
query = f"""
SELECT *
FROM dim_activity_times
WHERE at_id = {at_id};
"""
df_at = data_from_data_sink(query)

In [ ]:
query = f"""
SELECT 
   fl_workouts_pw,
   fl_cardio_min_pw,
   pa_t1,
    pa_t2,
    pa_t3,
    pa_t4,
    fitness_type_endurence,
    fitness_type_strength,
    fitness_type_flexibility,
    fitness_type_relaation,
    motivation_type,
    sc_strava_social_score,
    sc_strava_event_dog
FROM dim_strava
WHERE user_number = {user}
Order by entity_id DESC
LIMIT 1;
"""
df_strava = data_from_data_sink(query)

In [ ]:
df_user = pd.concat([df_health.reset_index(drop = True),df_strava.reset_index(drop = True )], axis=1)

In [ ]:
print(df_user)

In [ ]:

# Sortierung vorbereiten (vor dem Pivot!)
order_day_part = [
    "Early Morning (4–8)",
    "Morning (8–12)",
    "Noon (12–16)",
    "Afternoon (16–20)",
    "Evening (20–24)",
    "Night (0–4)"
]
df_at = df_at.pivot(index="day_part", columns="weekday", values="sleep_hours")
print(df_at)

In [ ]:


# Pivot-Tabelle erstellen (nach Sortierung)
order_day_part = [
    "Early Morning\n(4–8)",
    "Morning\n(8–12)",
    "Noon\n(12–16)",
    "Afternoon\n(16–20)",
    "Evening\n(20–24)",
    "Night\n(0–4)"
]

heatmap_data = df_at.loc[order_day_part]


# Neue Wochentagsbeschriftungen
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_labels = ["MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN"]
heatmap_data = heatmap_data[weekday_order]

# Bildgröße (8.4 cm x 4 cm in inch)
figsize_inch = (20 / 2.54, 7 / 2.54)

# Plot
plt.figure(figsize=figsize_inch, dpi=300)
sns.set_theme(style="white")

ax = sns.heatmap(
    heatmap_data,
    cmap="coolwarm",
    annot=False,
    linewidths=0.3,
    linecolor='white',
    cbar=False,  # Colorbar deaktivieren
    xticklabels=weekday_labels,
    yticklabels=False  # Y-Labels ausblenden
)

# Titel & Achsentexte
plt.title("", fontsize=0, weight='bold')
plt.ylabel("Daytime", fontsize=15, fontweight='bold')
plt.xlabel("")  # kein X-Achsentitel
plt.xticks(rotation=0, fontsize=15, fontweight='bold')
plt.yticks(fontsize=4)



plt.tight_layout()
plt.savefig("images/charts/activity_heatmap.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np


# Werte definieren

avg = df_user["sleep_hours_avg"][0]   # tatsächlicher Schlaf in Stunden
goal = df_user["sleep_hours_goal"][0]  
up_std =avg+ df_user["sleep_hours_std"][0]      # empfohlene Schlafdauer
down_std =avg-df_user["sleep_hours_std"][0] 
fig, ax = plt.subplots(figsize=(6, 1.5))

gradient = np.linspace(0, 1, 100).reshape(1, -1)

if avg > goal:
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#65cbc6", "#66cb92"])
else: 
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#e2545b", "#e16549"])

ax.imshow(
    gradient,
    extent=[0, avg, 0.15, 0.45],  # [x_start, x_end, y_start, y_end]
    origin='lower',
    aspect='auto',
    cmap=cmap
)

# Optional: Rahmen um den Verlauf
ax.barh(
    y=0.3,
    width=avg,
    height=0.3,
    edgecolor='black',
    facecolor='none',
    linewidth=1.0
)
# Achsenlimits
ax.set_xlim(0, 10)
ax.set_ylim(-0.5, 1)

# Y-Achse komplett entfernen
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

# X-Achsenticks als Uhrzeit
ticks = list(range(0, 11))
tick_labels = [f"{h:02d}:00" for h in ticks]
ax.set_xticks(ticks)
ax.set_xticklabels(tick_labels, fontsize=10, fontweight='bold')

# Kein Achsentitel
ax.set_xlabel("")  # oder: weglassen
ax.axvline(goal, color='black', linestyle='--', linewidth=1)
ax.axvline(up_std, color='red', linestyle=':', linewidth=1)
ax.axvline(down_std, color='purple', linestyle=':', linewidth=1)
ax.text(
    down_std ,   # leicht rechts neben der Linie
    0.75,            # vertikal mittig
    "minimum",  # dein Labeltext
    va='center',    # vertikale Ausrichtung
    ha='center',      # horizontale Ausrichtung
    fontsize=12,
    color='black',
    bbox=dict(
        facecolor='white',
        edgecolor='none',
        boxstyle='round,pad=0.2'
    )
    
)

ax.text(
    goal ,   # leicht rechts neben der Linie
    0.75,            # vertikal mittig
    "goal",  # dein Labeltext
    va='center',    # vertikale Ausrichtung
    ha='center',      # horizontale Ausrichtung
    fontsize=12,
    color='black',
    bbox=dict(
        facecolor='white',
        edgecolor='none',
        boxstyle='round,pad=0.2'
    )
    
)

if (up_std < 10):
    ax.text(
        up_std ,   # leicht rechts neben der Linie
        0.75,            # vertikal mittig
        "maximum",  # dein Labeltext
        va='center',    # vertikale Ausrichtung
        ha='center',      # horizontale Ausrichtung
        fontsize=12,
        color='black',
        bbox=dict(
            facecolor='white',
            edgecolor='none',
            boxstyle='round,pad=0.2'
        )
        
    )
# Untere Linie (Achsenlinie unten) entfernen
ax.spines['bottom'].set_visible(False)
ax.spines['top'].set_visible(False)

# Optional: Leichte Gitterlinie zur besseren Lesbarkeit
ax.grid(axis='x', linestyle=':', linewidth=0.3, color='gray', alpha=0.4)

plt.tight_layout()
plt.savefig("images/charts/sleep_chart.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Beispielwerte für drei Balken
balken = [
    {"value": df_user["steps_avg"][0], "goal": 10000, "label": "Steps per Day"},
    {"value": df_user["fl_workouts_pw"][0]/7, "goal": 1, "label": "Workouts per Week"},
    {"value": df_user["fl_cardio_min_pw"][0], "goal": 120,  "label": "h Cardio per Week"}
]

fig, axs = plt.subplots(nrows=3, figsize=(6, 2.5), constrained_layout=True)
i = 0
for ax, data in zip(axs, balken):
    value = data["value"]
    goal = data["goal"]
    xlim = (0, max(value, goal) * 1.2)  # Dynamische X-Achsenbegrenzung

    if value >= goal:
        cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#65cbc6", "#66cb92"])
    else: 
        cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#e2545b", "#e16549"])


    # Parameter für Balkenhöhe und y-Position (zentriert)
    bar_height = 0.4
    bar_y = 0.5 - bar_height / 2



    # Farbverlauf
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(
        gradient,
        extent=[0, value, bar_y, bar_y + bar_height],
        origin='lower',
        aspect='auto',
        cmap=cmap)

    # Umrandung
    ax.barh(y=bar_y + bar_height / 2, width=value, height=bar_height, color='none', edgecolor="black")
    # Ziel- und Mindestlinie
    ax.axvline(goal, color='black', linestyle='--', linewidth=1)
    ax.text(goal, bar_y + bar_height + 0.05, "goal", ha='center', va='bottom', fontsize=8,bbox=dict(
        facecolor="#66cb92",
        edgecolor='none',
        boxstyle='round,pad=0.2'
    ))
    # Achsen konfigurieren
    ax.set_xlim(*xlim)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel(data["label"], fontsize= 15, fontweight='bold')
    ticks = np.linspace(xlim[0], xlim[1], 6)
    ax.grid(axis='x', linestyle=':', linewidth=1, color='gray', alpha=.5)
    i = i +1
    if i ==1:  # Ganzzahlen mit Tausenderpunkt
        labels = [f"{x:,}".replace(",", ".") for x in ticks]
    elif i == 2:
        labels = [f"{round(x,1):,}".replace(",", ".") for x in ticks]
    elif i == 3:  # Minuten → hh:mm Format
        labels = [f"{int(x)//60}:{int(x)%60:02d}" for x in ticks]

    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, fontsize=8)

    # Rahmenlinien entfernen
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.savefig("images/charts/fitness_level.png", dpi=100, bbox_inches='tight')
plt.close()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

fig, ax = plt.subplots(figsize=(6, 1.5))
# Werte definieren

value = df_user["sc_strava_social_score"][0] *100  # tatsächlicher Schlaf in Stunden
print(value)

gradient = np.linspace(0, 1, 100).reshape(1, -1)

if value > 30:
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#65cbc6", "#66cb92"])
else: 
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#e2545b", "#e16549"])

ax.imshow(
    gradient,
    extent=[0, value, 0.15, 0.45],  # [x_start, x_end, y_start, y_end]
    origin='lower',
    aspect='auto',
    cmap=cmap
)

# Optional: Rahmen um den Verlauf
ax.barh(
    y=0.3,
    width=value,
    height=0.3,
    edgecolor='black',
    facecolor='none',
    linewidth=1.0
)
# Achsenlimits
ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 1)

# Y-Achse komplett entfernen
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

# X-Achsenticks als Uhrzeit
ticks = list(range(0, 100,10))
tick_labels = [f"{h}%" for h in ticks]
ax.set_xticks(ticks)
ax.set_xticklabels(tick_labels, fontsize=10, fontweight='bold')

# Kein Achsentitel
ax.set_title("Social Score", fontsize=15, fontweight='bold')    
ax.set_xlabel("")  # oder: weglassen
# ax.axvline(goal, color='black', linestyle='--', linewidth=1)
# ax.axvline(up_std, color='red', linestyle=':', linewidth=1)
# ax.axvline(down_std, color='purple', linestyle=':', linewidth=1)
# ax.text(
#     down_std ,   # leicht rechts neben der Linie
#     0.75,            # vertikal mittig
#     "minimum",  # dein Labeltext
#     va='center',    # vertikale Ausrichtung
#     ha='center',      # horizontale Ausrichtung
#     fontsize=12,
#     color='black',
#     bbox=dict(
#         facecolor='white',
#         edgecolor='none',
#         boxstyle='round,pad=0.2'
#     )
    
# )



# Untere Linie (Achsenlinie unten) entfernen
ax.spines['bottom'].set_visible(False)
ax.spines['top'].set_visible(False)

# Optional: Leichte Gitterlinie zur besseren Lesbarkeit
ax.grid(axis='x', linestyle=':', linewidth=0.3, color='gray', alpha=0.4)

plt.tight_layout()
plt.savefig("images/charts/social_score.png", dpi=300, bbox_inches='tight')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

fig, ax = plt.subplots(figsize=(12, 1.5))
# Werte definieren

value = df_user["tot_wellness_score"][0] *100  # tatsächlicher Schlaf in Stunden
print(value)

gradient = np.linspace(0, 1, 100).reshape(1, -1)

if value > 50:
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#65cbc6", "#66cb92"])
else: 
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_gradient", ["#e2545b", "#e16549"])

ax.imshow(
    gradient,
    extent=[0, value, 0.15, 0.45],  # [x_start, x_end, y_start, y_end]
    origin='lower',
    aspect='auto',
    cmap=cmap
)

# Optional: Rahmen um den Verlauf
ax.barh(
    y=0.3,
    width=value,
    height=0.3,
    edgecolor='black',
    facecolor='none',
    linewidth=1.0
)
# Achsenlimits
ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 1)

# Y-Achse komplett entfernen
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

# X-Achsenticks als Uhrzeit
ticks = list(range(0, 100,10))
tick_labels = [f"{h}%" for h in ticks]
ax.set_xticks(ticks)
ax.set_xticklabels(tick_labels, fontsize=10, fontweight='bold')

# Kein Achsentitel
ax.set_title("Wellness Score", fontsize=15, fontweight='bold')    
ax.set_xlabel("")  # oder: weglassen
# ax.axvline(goal, color='black', linestyle='--', linewidth=1)
# ax.axvline(up_std, color='red', linestyle=':', linewidth=1)
# ax.axvline(down_std, color='purple', linestyle=':', linewidth=1)
# ax.text(
#     down_std ,   # leicht rechts neben der Linie
#     0.75,            # vertikal mittig
#     "minimum",  # dein Labeltext
#     va='center',    # vertikale Ausrichtung
#     ha='center',      # horizontale Ausrichtung
#     fontsize=12,
#     color='black',
#     bbox=dict(
#         facecolor='white',
#         edgecolor='none',
#         boxstyle='round,pad=0.2'
#     )
    
# )



# Untere Linie (Achsenlinie unten) entfernen
ax.spines['bottom'].set_visible(False)
ax.spines['top'].set_visible(False)

# Optional: Leichte Gitterlinie zur besseren Lesbarkeit
ax.grid(axis='x', linestyle=':', linewidth=0.3, color='gray', alpha=0.4)

plt.tight_layout()
plt.savefig("images/charts/wellness_score.png", dpi=300, bbox_inches='tight')


In [ ]:
user_secrets= pd.read_csv("data/secret_info.csv")
query = f"""
SELECT user_id FROM dim_user WHERE user_number = {user}
"""
user_id = data_from_data_sink(query).iloc[0, 0]

user_secrets.loc[user_secrets["user_id"] == user_id,"date_of_birth"].iloc[0]

print(user_secrets)

In [ ]:
import time
from datetime import datetime   

f_name = user_secrets.loc[user_secrets["user_id"] == user_id,"first_name"].iloc[0]
l_name = user_secrets.loc[user_secrets["user_id"] == user_id,"last_name"].iloc[0]
var_name = f"{f_name} {l_name}"
# 1. Geburtsdatum
var_date_birth =user_secrets.loc[user_secrets["user_id"] == user_id,"date_of_birth"].iloc[0]
dob = datetime.strptime(var_date_birth, "%d.%m.%Y")
# 2. Alter berechnen
today = datetime.today()
var_age = (today - dob).days // 365
var_place_birth = user_secrets.loc[user_secrets["user_id"] == user_id,"place_of_birth"].iloc[0]
street = user_secrets.loc[user_secrets["user_id"] == user_id,"street"].iloc[0]
number = user_secrets.loc[user_secrets["user_id"] == user_id,"number"].iloc[0]
var_street_number = f"{street} {number}"
city= user_secrets.loc[user_secrets["user_id"] == user_id,"city"].iloc[0]
pa_1 =df_user["pa_t1"][0].split("\n")[0]
pa_2 = df_user["pa_t2"][0].split("\n")[0]
pa_3 = df_user["pa_t3"][0].split("\n")[0]
pa_4 = df_user["pa_t4"][0].split("\n")[0]
print(pa_1, pa_2, pa_3, pa_4)   
var_occ_pa_2 = "opacity: 1;" if pa_2 == "None" else "opacity: 1;"
var_occ_pa_3 = "opacity: 1;" if pa_2 == "None" else "opacity: 1;"
var_occ_pa_4 = "opacity: 1;" if pa_2 == "None" else "opacity: 1;" 
var_wake_ups = round(df_user["sleep_wakeups_avg"][0],0)
var_motivation = df_user["motivation_type"][0].split("\n")[0].lower()
var_event_dog_true =df_user["sc_strava_event_dog"][0] 
var_event_dog = "event_dog" if var_event_dog_true else "no_event_dog"

In [ ]:
user_vars = {
    "var_name": var_name,
    "var_date_birth": var_date_birth,
    "var_age": var_age,
    "var_place_birth": var_place_birth,
    "var_street_number": "Stephanstraße 3", #var_street_number,
    "city": city,
    "pa_1": pa_1,
    "pa_2": pa_2,
    "pa_3": pa_3,
    "pa_4": pa_4,
    "var_occ_pa_2": var_occ_pa_2,
    "var_occ_pa_3": var_occ_pa_3,
    "var_occ_pa_4": var_occ_pa_4,
    "var_wake_ups": var_wake_ups,
    "var_motivation": var_motivation,
    "var_event_dog_true": var_event_dog_true,
    "var_event_dog": var_event_dog,
}

In [ ]:
from jinja2 import Environment, FileSystemLoader
import os
output_file = f"rendered_profiles/health_{f_name}_{l_name}.html"
env = Environment(loader=FileSystemLoader("templates"))
template = env.get_template("health_tmpl.html")

# Rendern
output = template.render(user_vars)
os.makedirs("rendered_profiles", exist_ok=True)
#env = Environment(loader=FileSystemLoader("rendered_profiles"))
with open(output_file, "w", encoding="utf-8") as f:
    f.write(output)

print(f"✅ Template erfolgreich gerendert als {output_file}")